# RMIT_SGS_HCMUT NeuroX - C1 final submission

**Chosen model: mistralai/Mistral-7B-Instruct-v0.3 (approved list), LoRA sequence classification**

## Approach
- Task: 3-class prompt-safety classification (BENIGN / JAILBREAK / PROMPT_INJECTION), multilingual (en/fr/it/pt).
- Model: `mistralai/Mistral-7B-Instruct-v0.3` + LoRA (r=16, alpha=32, all-linear, trainable `score` head), sequence-classification head, max_len 512 (head+tail token truncation), 2 epochs, lr 5e-5, sqrt class weights, bf16.
- **5-fold CV runs with different seeds / recipes** (`mistral7b_lora_v2`, `_v3`, `_v4`): each gives OOF predictions and 5 fold-model test predictions.
- **Pseudo-labelling**: our own v2 teacher (5-fold average) labels the unlabeled test text; test rows with confidence >= 0.95 are added to training for the `mistral_pl95` fold run and for the full-fit seeds.
- **Full-fit seeds**: models trained on 100% of train: seeds 21/22/23 with pseudo-labels (`mistral_full_pl`) and seeds 31/32 without (`mistral_full`).
- **Blend**: geometric mean (mean of log-probs, then softmax). Each fold run (its 5 folds averaged first) has weight 1.0, each full-fit seed weight 0.6. Full-fit seeds must pass sanity gates (class distribution within 1.5pt of the prior, argmax agreement >= 0.975 with the fold blend). **No thresholds / no class-bias tuning.**
- Results: 4-run fold-blend **OOF macro-F1 = 0.9821**, public leaderboard **0.98974**.

Source code: https://github.com/NaNaBoo311/RMIT_HACKATHON_2026 (final pipeline commit `06e65ba`).

## Provenance / proof of training (Modal, not Kaggle)
- **Provenance report** (Modal app IDs + URLs for every run, per-fold metrics and train times, GPU, billing, Kaggle submission history, git timeline): https://github.com/NaNaBoo311/RMIT_HACKATHON_2026/blob/main/docs/C1_PROVENANCE.md
- **Modal dashboard screenshots**: https://github.com/NaNaBoo311/RMIT_HACKATHON_2026/tree/main/docs/evidence
- **Training code**: [`core-ai/modal_app/train.py`](https://github.com/NaNaBoo311/RMIT_HACKATHON_2026/blob/main/core-ai/modal_app/train.py), full pipeline [`core-ai/scripts/reproduce_final.sh`](https://github.com/NaNaBoo311/RMIT_HACKATHON_2026/blob/main/core-ai/scripts/reproduce_final.sh), reproducibility notes in [`core-ai/README.md`](https://github.com/NaNaBoo311/RMIT_HACKATHON_2026/blob/main/core-ai/README.md)
- **Saved predictions used here** (Kaggle dataset): https://www.kaggle.com/datasets/nostidus/rmit-c1-mistral-final-preds
- Training ran on Modal A100-80GB (~25 GPU-h, 3 team workspaces). This notebook does **not** train; it re-runs the final blend from the saved predictions and checks that the output is byte-identical (sha256) to our final Kaggle submission.

**Note on reproducibility:** training ran on Modal (A100-80GB, ~25 GPU-hours total), so it cannot be repeated inside a Kaggle session. This notebook (internet OFF, CPU is enough) loads the saved per-fold / per-seed prediction arrays from the attached dataset `rmit-c1-mistral-final-preds`, recomputes the OOF score and re-creates the exact final `submission.csv` (sha256-checked). A single-fold training function is included in the appendix (`RUN_TRAINING = False` by default; demo only).

In [ ]:
import os, glob, hashlib
import numpy as np, pandas as pd
from pathlib import Path

LABELS = ["BENIGN", "JAILBREAK", "PROMPT_INJECTION"]
EXPECTED_SHA = "125781c7d933dd24a0907cf17f65321c5efb3b85ec5027be2d151d1baa603b18"

# --- competition data (override with env RMIT_COMP_DIR for local runs)
cands = [os.environ.get("RMIT_COMP_DIR", ""), "/kaggle/input/rmit-hackathon-2026"]
COMP = next((Path(c) for c in cands if c and (Path(c) / "sample_submission.csv").exists()), None)
if COMP is None:
    hits = glob.glob("/kaggle/input/**/sample_submission.csv", recursive=True)
    assert hits, "competition data not found"
    COMP = Path(hits[0]).parent

# --- prediction dataset (override with env RMIT_DS_DIR); folder containing mistral7b_lora_v2
env_ds = os.environ.get("RMIT_DS_DIR", "")
if env_ds and (Path(env_ds) / "mistral7b_lora_v2").exists():
    DS = Path(env_ds)
else:
    hits = glob.glob("/kaggle/input/**/mistral7b_lora_v2", recursive=True)
    assert hits, "prediction dataset not found (attach rmit-c1-mistral-final-preds)"
    DS = Path(hits[0]).parent

WORK = Path(os.environ.get("RMIT_WORK_DIR", "/kaggle/working")); WORK.mkdir(parents=True, exist_ok=True)
print("competition:", COMP, "\ndataset:", DS, "\nwork:", WORK)
train = pd.read_csv(COMP / "train.csv"); test = pd.read_csv(COMP / "test.csv"); sample = pd.read_csv(COMP / "sample_submission.csv")
print(train.shape, test.shape, sample.shape)

In [ ]:
FOLD_RUNS = ["mistral7b_lora_v2", "mistral7b_lora_v3", "mistral7b_lora_v4", "mistral_pl95"]
FULL_RUNS = ["mistral_full_pl", "mistral_full"]

def macro_f1(y, pred, k=3):
    cm = np.bincount(y * k + pred, minlength=k * k).reshape(k, k)
    tp = np.diag(cm).astype(float); denom = cm.sum(0) + cm.sum(1)
    return float(np.mean(np.where(denom > 0, 2 * tp / np.maximum(denom, 1), 0.0)))

def combine(arrs, w, avg="geo"):
    # identical to rmit_hack.ensemble.combine
    w = np.asarray(w, dtype=float); w = w / w.sum()
    if avg == "prob":
        return sum(wi * a for wi, a in zip(w, arrs))
    lg = sum(wi * np.log(np.clip(a, 0, 1) + 1e-9) for wi, a in zip(w, arrs))
    lg = lg - lg.max(-1, keepdims=True)
    e = np.exp(lg)
    return e / e.sum(-1, keepdims=True)

y = train["label"].map({l: i for i, l in enumerate(LABELS)}).to_numpy()
id2pos = {i: p for p, i in enumerate(train["id"])}

def load_oof(run):
    oof = np.full((len(train), 3), np.nan)
    for fd in sorted((DS / run).glob("fold*")):
        df = pd.read_csv(fd / "oof.csv")
        oof[df["id"].map(id2pos).to_numpy()] = df[[f"p_{l}" for l in LABELS]].to_numpy()
    assert not np.isnan(oof).any(), f"{run}: OOF incomplete"
    return oof

oofs = {r: load_oof(r) for r in FOLD_RUNS}
for r, o in oofs.items():
    print(f"{r:20s} OOF macro-F1 = {macro_f1(y, o.argmax(-1)):.4f}")
blend_oof = combine(list(oofs.values()), np.ones(len(oofs)), "geo")
oof_f1 = macro_f1(y, blend_oof.argmax(-1))
print(f"{'geo-mean of 4 fold runs':20s} OOF macro-F1 = {oof_f1:.4f}   (expected 0.9821)")

In [ ]:
import hashlib
PRIOR = np.array([0.62, 0.31, 0.07]); DIST_TOL, AGREE_MIN, AGREE_WARN = 0.015, 0.975, 0.985; FULL_W = 0.6
tid = test["id"].to_numpy()

def _load(p):
    tp = np.load(p / "test_probs.npy")
    assert tp.shape == (len(tid), 3), f"{p}: bad shape {tp.shape}"
    assert (np.load(p / "test_ids.npy", allow_pickle=True) == tid).all(), f"{p}: test_ids order != test.csv"
    return tp

full, folds = [], []
for r in FULL_RUNS:   # same order as the CLI: --full outputs/mistral_full_pl outputs/mistral_full
    dirs = sorted(d for d in (DS / r).glob("full_s*") if (d / "test_probs.npy").exists())
    assert dirs, r
    full += [(f"{r}/{d.name}", _load(d), FULL_W) for d in dirs]
for r in FOLD_RUNS:
    dirs = sorted(d for d in (DS / r).glob("fold*") if (d / "test_probs.npy").exists())
    folds.append((f"{r}[{len(dirs)}f]", combine([_load(d) for d in dirs], np.ones(len(dirs)), "geo"), 1.0))

# sanity gates for full-fit seeds vs fold-avg blend
ref = combine([p for _, p, _ in folds], [w for _, _, w in folds], "geo").argmax(-1)
kept = []
for n, p, w in full:
    pr = p.argmax(-1); dist = np.bincount(pr, minlength=3) / len(pr); agree = float((pr == ref).mean())
    ok = bool((np.abs(dist - PRIOR) <= DIST_TOL).all()) and agree >= AGREE_MIN
    print(f"gate {n}: dist={dist.round(3).tolist()} agree={agree:.4f}", "PASS" if ok else "FAIL - EXCLUDED",
          "WARN agreement < 0.985" if agree < AGREE_WARN else "")
    if ok: kept.append((n, p, w))
models = kept + folds
print("models:", [(n, w) for n, _, w in models])

P = combine([m for _, m, _ in models], [w for _, _, w in models], "geo")
pred = np.log(np.clip(P, 1e-9, 1)).argmax(-1)          # no bias, no thresholds
mapping = dict(zip(tid, [LABELS[i] for i in pred]))
sub = sample.copy()
sub[sub.columns[-1]] = sub[sub.columns[0]].map(mapping)
assert len(sub) == 6000 == len(test) and sub[sub.columns[-1]].isin(LABELS).all() and set(sub[sub.columns[0]]) == set(tid)
out = WORK / "submission.csv"; sub.to_csv(out, index=False)
print(sub[sub.columns[-1]].value_counts(normalize=True).round(3).to_dict())
sha = hashlib.sha256(out.read_bytes()).hexdigest()
print("sha256:", sha)
if sha == EXPECTED_SHA:
    print("OK: identical to the final Kaggle submission (FINAL_mistral_full.csv, public LB 0.98974)")
else:
    print("WARNING: sha256 differs from the submitted file (float/library differences?).")
    ref_path = next(iter(glob.glob(str(COMP / "**" / "FINAL_mistral_full.csv"), recursive=True)), None)
    if ref_path:
        r = pd.read_csv(ref_path).set_index(sub.columns[0])[sub.columns[-1]].reindex(sub[sub.columns[0]]).values
        print("agreement with submitted file:", float((r == sub[sub.columns[-1]].values).mean()))

## Appendix: training code (optional, not executed by default)

`train_fold` below is the single-fold version of the function used on Modal (`modal_app/train.py` in the repo): Mistral-7B-Instruct-v0.3 + LoRA (`all-linear`, `modules_to_save=["score"]`), sqrt class weights, head+tail truncation to 512 tokens, 2 epochs, lr 5e-5, effective batch 16, gradient checkpointing, optional pseudo-labels. Predictions are computed with our own sequential loop (not `Trainer.predict`, which is shuffled by `group_by_length`).

To run one fold on Kaggle:
1. Accelerator **GPU T4 x2** (or P100), internet is not needed if the model is attached.
2. Add the model via *Add Input -> Models*: **aisuko/mistral-7b-instruct-v0.3**; set `MODEL_PATH` below to its folder.
3. A T4 has no bf16 and 16 GB, so use `LOAD_4BIT = True` (QLoRA, needs `bitsandbytes`/`peft` installed) and reduce `MAX_LEN` / `batch_size`. Expect several hours per fold; the original runs used full bf16 on an A100-80GB (~1 h/fold) so numbers will differ slightly.
4. Set `RUN_TRAINING = True`.

In [ ]:
RUN_TRAINING = False
MODEL_PATH = "/kaggle/input/mistral-7b-instruct-v0.3/pytorch/default/1"   # adjust to the attached Kaggle Model folder
FOLD = 0
LOAD_4BIT = True      # required on T4 (16 GB, no bf16)
CFG = dict(max_len=512, epochs=2, lr=5e-5, batch_size=4, grad_accum=4, warmup_ratio=0.1, weight_decay=0.01,
           lora_r=16, lora_alpha=32, lora_dropout=0.05, seed=42, pseudo_threshold=0.95, pseudo_run=None)  # pseudo_run: e.g. a dir of fold*/test_probs.npy

def train_fold(cfg, fold, model_path, folds_csv, load_4bit=True):
    import torch, numpy as np, pandas as pd
    from sklearn.metrics import f1_score
    from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
                              Trainer, TrainingArguments, set_seed, BitsAndBytesConfig)
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    set_seed(cfg["seed"])
    label2id = {l: i for i, l in enumerate(LABELS)}
    tr = train.merge(pd.read_csv(folds_csv), on="id", how="left"); tr["text"] = tr["text"].fillna("").astype(str)
    te = test.copy(); te["text"] = te["text"].fillna("").astype(str)
    y_all = tr["label"].map(label2id).to_numpy()
    tr_idx = np.where(tr["fold"].to_numpy() != fold)[0]; va_idx = np.where(tr["fold"].to_numpy() == fold)[0]

    tok = AutoTokenizer.from_pretrained(model_path)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    probe = tok("hello world", add_special_tokens=True)["input_ids"]; plain = tok("hello world", add_special_tokens=False)["input_ids"]
    pos = next(i for i in range(len(probe) - len(plain) + 1) if probe[i:i + len(plain)] == plain)
    prefix, suffix = probe[:pos], probe[pos + len(plain):]
    keep = cfg["max_len"] - len(prefix) - len(suffix); head_n = min(keep, max(keep - 128, keep // 2))
    def encode(texts):   # head + tail token truncation
        out = []
        for ids in tok(texts, add_special_tokens=False, truncation=False)["input_ids"]:
            if len(ids) > keep: ids = ids[:head_n] + ids[len(ids) - (keep - head_n):]
            out.append(prefix + ids + suffix)
        return out

    class DS_(torch.utils.data.Dataset):
        def __init__(self, texts, ys=None, sort=False):
            self.ids = encode(texts); self.ys = ys
            self.order = np.argsort([len(x) for x in self.ids], kind="stable") if sort else np.arange(len(self.ids))
            self.inv = np.argsort(self.order)
        def __len__(self): return len(self.ids)
        def __getitem__(self, i):
            j = int(self.order[i]); item = {"input_ids": self.ids[j], "attention_mask": [1] * len(self.ids[j])}
            if self.ys is not None: item["labels"] = int(self.ys[j])
            return item

    tr_texts = tr["text"].iloc[tr_idx].tolist(); tr_y = y_all[tr_idx]
    if cfg.get("pseudo_run"):   # pseudo-labels from a teacher run: avg of fold*/test_probs.npy, keep confidence >= threshold
        pdir = Path(cfg["pseudo_run"]); pf = sorted(pdir.glob("fold*/test_probs.npy"))
        pp = np.mean([np.load(f) for f in pf], axis=0); sel = pp.max(-1) >= cfg["pseudo_threshold"]
        tr_texts += te["text"].iloc[np.where(sel)[0]].tolist(); tr_y = np.concatenate([tr_y, pp[sel].argmax(-1)])
    ds_tr = DS_(tr_texts, tr_y); ds_va = DS_(tr["text"].iloc[va_idx].tolist(), y_all[va_idx], sort=True); ds_te = DS_(te["text"].tolist(), sort=True)

    bf16 = torch.cuda.is_bf16_supported()
    kw = dict(num_labels=3, id2label=dict(enumerate(LABELS)), label2id=label2id, ignore_mismatched_sizes=True,
              torch_dtype=torch.bfloat16 if bf16 else torch.float16)
    if load_4bit:
        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16 if bf16 else torch.float16, bnb_4bit_use_double_quant=True)
        kw["device_map"] = {"": 0}
    model = AutoModelForSequenceClassification.from_pretrained(model_path, **kw)
    model.config.pad_token_id = tok.pad_token_id
    if load_4bit: model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    else: model.enable_input_require_grads()
    model = get_peft_model(model, LoraConfig(task_type="SEQ_CLS", r=cfg["lora_r"], lora_alpha=cfg["lora_alpha"],
                           lora_dropout=cfg["lora_dropout"], target_modules="all-linear", modules_to_save=["score"]))
    for p in model.parameters():
        if p.requires_grad: p.data = p.data.float()     # fp32 master weights for trainable params
    model.print_trainable_parameters()

    counts = np.bincount(tr_y, minlength=3).astype(float); cw_t = torch.tensor((counts.max() / counts) ** 0.5, dtype=torch.float)  # sqrt class weights
    class WTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kw):
            labels_ = inputs.pop("labels"); out = model(**inputs)
            loss = torch.nn.functional.cross_entropy(out.logits.float(), labels_, weight=cw_t.to(out.logits.device))
            return (loss, out) if return_outputs else loss
    args = TrainingArguments(output_dir="/kaggle/working/ckpt", num_train_epochs=cfg["epochs"], learning_rate=cfg["lr"],
        per_device_train_batch_size=cfg["batch_size"], gradient_accumulation_steps=cfg["grad_accum"], warmup_ratio=cfg["warmup_ratio"],
        weight_decay=cfg["weight_decay"], save_strategy="no", bf16=bf16, fp16=not bf16, max_grad_norm=1.0, logging_steps=50,
        report_to=[], seed=cfg["seed"], group_by_length=True, gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False}, remove_unused_columns=False)
    trainer = WTrainer(model=model, args=args, train_dataset=ds_tr, data_collator=DataCollatorWithPadding(tok))
    trainer.train()

    model.eval(); dev = next(model.parameters()).device; coll = DataCollatorWithPadding(tok)
    def predict_probs(ds, bs=8):   # own sequential loop (Trainer.predict is shuffled with group_by_length)
        outs = []
        with torch.no_grad():
            for b in torch.utils.data.DataLoader(ds, batch_size=bs, shuffle=False, collate_fn=coll):
                b.pop("labels", None); b = {k: v.to(dev) for k, v in b.items()}
                with torch.autocast("cuda", dtype=torch.bfloat16 if bf16 else torch.float16):
                    outs.append(model(**b).logits.float().cpu().numpy())
        x = np.concatenate(outs).astype(np.float64); x -= x.max(-1, keepdims=True); e = np.exp(x)
        return (e / e.sum(-1, keepdims=True))[ds.inv]
    va_probs, te_probs = predict_probs(ds_va), predict_probs(ds_te)
    print(f"fold {fold} OOF macro-F1 = {f1_score(y_all[va_idx], va_probs.argmax(-1), average='macro'):.4f}")
    out = Path("/kaggle/working") / f"fold{fold}"; out.mkdir(parents=True, exist_ok=True)
    np.save(out / "test_probs.npy", te_probs); np.save(out / "test_ids.npy", te["id"].to_numpy())
    pd.DataFrame(va_probs, columns=[f"p_{l}" for l in LABELS]).assign(id=tr["id"].iloc[va_idx].to_numpy())[["id"] + [f"p_{l}" for l in LABELS]].to_csv(out / "oof.csv", index=False)
    return va_probs, te_probs

if RUN_TRAINING:
    train_fold(CFG, FOLD, MODEL_PATH, DS / "folds.csv", LOAD_4BIT)
else:
    print("RUN_TRAINING=False: training skipped (predictions from the attached dataset were used above).")